In [1]:
import sys 
from pathlib import Path

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data" 

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv(DATA_DIR / 'processed_data.csv')

In [4]:
inputs=[
        
        'acc_x_u1', 'acc_x_u2', 'acc_x_u3', 'acc_x_u4', 'acc_x_u5', 
        'acc_y_u1', 'acc_y_u2', 'acc_y_u3', 'acc_y_u4', 'acc_y_u5', 
        'acc_z_u1', 'acc_z_u2', 'acc_z_u3', 'acc_z_u4', 'acc_z_u5',
        'gyr_x_u1', 'gyr_x_u2', 'gyr_x_u3', 'gyr_x_u4', 'gyr_x_u5', 
        'gyr_y_u1', 'gyr_y_u2', 'gyr_y_u3', 'gyr_y_u4', 'gyr_y_u5',
        'gyr_z_u1', 'gyr_z_u2', 'gyr_z_u3', 'gyr_z_u4', 'gyr_z_u5', 
        'mag_x_u1', 'mag_x_u2', 'mag_x_u3', 'mag_x_u4', 'mag_x_u5', 
        'mag_y_u1', 'mag_y_u2', 'mag_y_u3', 'mag_y_u4', 'mag_y_u5', 
        'mag_z_u1', 'mag_z_u2', 'mag_z_u3', 'mag_z_u4', 'mag_z_u5',
]


In [5]:
exercises = ['e1','e2','e3','e4','e5','e6','e7','e8']
subject=['s1', 's2', 's3', 's5']
labels=['correct', 'low_amplitude', 'fast']

for s in  subject:
    for e in exercises:
        for l in labels:


            df_r=df[(df['subject'] == s) & (df['exercise'] == e) & (df['label']==l)]

            
            duration = df_r['time index'].max() - df_r['time index'].min()

            repetition_duration= duration /10

            print(repetition_duration, duration, l)

            

202.8 2028 correct
192.3 1923 low_amplitude
84.2 842 fast
192.8 1928 correct
189.3 1893 low_amplitude
125.9 1259 fast
197.5 1975 correct
202.4 2024 low_amplitude
109.1 1091 fast
204.2 2042 correct
173.6 1736 low_amplitude
101.4 1014 fast
182.1 1821 correct
185.7 1857 low_amplitude
91.2 912 fast
237.1 2371 correct
189.3 1893 low_amplitude
110.4 1104 fast
224.8 2248 correct
198.4 1984 low_amplitude
104.9 1049 fast
180.9 1809 correct
191.7 1917 low_amplitude
84.5 845 fast
157.8 1578 correct
147.5 1475 low_amplitude
82.4 824 fast
184.3 1843 correct
153.7 1537 low_amplitude
100.8 1008 fast
171.5 1715 correct
136.7 1367 low_amplitude
82.9 829 fast
146.2 1462 correct
124.4 1244 low_amplitude
84.1 841 fast
139.1 1391 correct
116.9 1169 low_amplitude
81.5 815 fast
144.3 1443 correct
120.0 1200 low_amplitude
78.4 784 fast
131.8 1318 correct
108.9 1089 low_amplitude
77.5 775 fast
120.4 1204 correct
106.8 1068 low_amplitude
74.6 746 fast
164.3 1643 correct
149.3 1493 low_amplitude
85.8 858 fast
17

In [6]:
label_to_idx = {
    'correct': 0,
    'low_amplitude': 1,
    'fast': 2
}

all_sequences = []
all_labels = []
all_subjects = []   
all_exercises = []  

for s in subject:
    for e in exercises:
        for l in labels:

            df_r = df[
                (df['subject'] == s) &
                (df['exercise'] == e) &
                (df['label'] == l)
            ].copy()

            df_r = df_r.sort_values('time index')

            duration = df_r['time index'].max() - df_r['time index'].min()
            rep_len = int(duration / 10)

            for i in range(10):
                rep = df_r.iloc[i * rep_len:(i + 1) * rep_len]

                X_rep = rep[inputs].to_numpy()   # (timesteps, features)

                all_sequences.append(X_rep)
                all_labels.append(label_to_idx[l])
                all_subjects.append(s)
                all_exercises.append(e)

max_len = max(seq.shape[0] for seq in all_sequences)

padded = []

for seq in all_sequences:
    seq = torch.tensor(seq, dtype=torch.float32)  # (time, features)
    seq = seq.transpose(0, 1)                     # (features, time)

    pad_len = max_len - seq.shape[1]
    seq = F.pad(seq, (0, pad_len))

    padded.append(seq)

X = torch.stack(padded)  # (N, features, max_len)
y = torch.tensor(all_labels, dtype=torch.long)

In [7]:
class CNNmodel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=input_dim,
            out_channels=32,
            kernel_size=5,
            padding=2
        )

        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.squeeze(-1)
        x = self.fc(x)
        return x
    
model = CNNmodel(input_dim=60, num_classes=3)

In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_subjects = ['s1', 's2', 's3']
val_subjects = ['s5']

train_indices = [i for i, s in enumerate(all_subjects) if s in train_subjects]
val_indices = [i for i, s in enumerate(all_subjects) if s in val_subjects]

X_train = X[train_indices]
y_train = y[train_indices]

X_val = X[val_indices]
y_val = y[val_indices]

# Convert to tensors if they are numpy arrays
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

torch.Size([720, 45, 237]) torch.Size([720])
torch.Size([240, 45, 237]) torch.Size([240])


C:\Users\aless\AppData\Local\Temp\ipykernel_11148\1402832706.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_11148\1402832706.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long)
C:\Users\aless\AppData\Local\Temp\ipykernel_11148\1402832706.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val = torch.tensor(X_val, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_11148\

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNNmodel(input_dim=45, num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [10]:
from sklearn.metrics import confusion_matrix


In [11]:
for epoch in range(100):

    # TRAIN
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # VALIDATION
    model.eval()
    correct = 0
    total = 0
    val_loss=0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            
            
            
            outputs = model(inputs)

            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            


    val_loss /= len(val_loader.dataset)
    val_acc = correct / total
    cm = confusion_matrix(all_labels, all_preds)
    print(cm)

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val LOSS: {val_loss:.4f}")

print("Finished Training")

[[67 13  0]
 [27 53  0]
 [20 60  0]]
Epoch 1 | Train Loss: 1.0986 | Val LOSS: 1.0464
[[78  2  0]
 [42 38  0]
 [34 45  1]]
Epoch 2 | Train Loss: 1.0203 | Val LOSS: 1.0123
[[67 13  0]
 [14 66  0]
 [16 56  8]]
Epoch 3 | Train Loss: 0.9778 | Val LOSS: 0.9780
[[61 19  0]
 [10 70  0]
 [11 51 18]]
Epoch 4 | Train Loss: 0.9377 | Val LOSS: 0.9428
[[63 15  2]
 [19 61  0]
 [ 9 38 33]]
Epoch 5 | Train Loss: 0.8913 | Val LOSS: 0.9058
[[66 11  3]
 [23 54  3]
 [16 33 31]]
Epoch 6 | Train Loss: 0.8484 | Val LOSS: 0.8621
[[64 12  4]
 [19 46 15]
 [ 7 22 51]]
Epoch 7 | Train Loss: 0.7955 | Val LOSS: 0.8228
[[61  9 10]
 [20 39 21]
 [15 20 45]]
Epoch 8 | Train Loss: 0.7572 | Val LOSS: 0.7984
[[62  7 11]
 [20 41 19]
 [12 19 49]]
Epoch 9 | Train Loss: 0.7065 | Val LOSS: 0.7661
[[57  9 14]
 [12 46 22]
 [ 4 20 56]]
Epoch 10 | Train Loss: 0.6513 | Val LOSS: 0.7458
[[55 11 14]
 [20 42 18]
 [ 4 20 56]]
Epoch 11 | Train Loss: 0.6187 | Val LOSS: 0.7258
[[53 11 16]
 [14 46 20]
 [ 2 17 61]]
Epoch 12 | Train Loss: 0.5

KeyboardInterrupt: 

In [ ]:

val_loss = 0.0

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        
val_loss /= len(val_loader.dataset)